# 04 — Independent bilateral SAM/Vicon validation

Uses **one reviewed synchronization file per camera**. SAM joint angles must be on their **original video timebase** (`time_ms`). Vicon reference angles must be exported independently from the C3D with their own `time_ms`. No joint-derived timing corrections are applied. Left = red; right = green. SAM = solid; Vicon = dashed.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / "src").is_dir() and (ROOT.parent / "src").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "src").is_dir():
    raise FileNotFoundError("Run Jupyter from the repository root or notebooks/")
sys.path.insert(0, str(ROOT))
from src.comparison_engine import compare_angles
from src.plotting import plot_joint_overlays, plot_joint_errors, plot_camera_comparison
DATA = ROOT / "lunge_sam_out"
RESULTS = ROOT / "results"


## Configuration
The expected filenames below are illustrative: change the paths to match the actual **raw, SAM-only** CSV exports. Never use CSVs containing `time_ms_aligned` as raw inputs. Each candidate must be manually reviewed and saved under the correct camera as `sync.json`.


In [ ]:
SUBJECT = "S9"
TRIAL = "lunge_01"
CAMERAS = ("cam0", "cam1")
METHOD = "geo"
STATIC_WINDOW_MS = None  # Set e.g. 1000 only if an actual standing interval is verified.

VICON_CSV = DATA / SUBJECT / TRIAL / "vicon_angles.csv"
ANGLE_FILES = {
    camera: DATA / f"{SUBJECT.lower()}_lunge_{camera}_sam_angles_raw.csv"
    for camera in CAMERAS
}
SYNC_FILES = {
    camera: RESULTS / SUBJECT / TRIAL / camera / "sync.json"
    for camera in CAMERAS
}

for camera in CAMERAS:
    print(camera, "SAM", ANGLE_FILES[camera].exists(), "sync", SYNC_FILES[camera].exists())
print("Independent Vicon reference", VICON_CSV.exists())


## Input QC
Reject prealigned files and embedded Vicon columns, and verify that bilateral data are genuinely present. Missing left-leg columns cannot be filled using the right leg.


In [ ]:
required_sam = [f"{side}_{joint}_flex_{METHOD}" for side in ("l", "r")
                for joint in ("hip", "knee", "ankle")]
required_vicon = [f"{side}_vicon_{joint}_flex" for side in ("l", "r")
                  for joint in ("hip", "knee", "ankle")]
if VICON_CSV.exists():
    vdf = pd.read_csv(VICON_CSV)
    print("Missing Vicon bilateral references:", sorted(set(required_vicon) - set(vdf.columns)))
for camera in CAMERAS:
    p = ANGLE_FILES[camera]
    if not p.exists():
        continue
    frame = pd.read_csv(p)
    if "time_ms" not in frame or "time_ms_aligned" in frame:
        raise ValueError(f"{p.name}: must have original time_ms, without time_ms_aligned")
    if any("vicon" in c.lower() for c in frame.columns):
        raise ValueError(f"{p.name}: contains embedded Vicon columns")
    print(camera, "missing SAM angle columns:", sorted(set(required_sam) - set(frame.columns)))


## Compare each camera and side
The existing `compare_angles()` writes `aligned_angles.csv` and `validation_metrics.csv` for each side. The outputs are **not** interpreted as validated unless the input `sync.json` was independently reviewed and the Vicon anatomical definitions are compatible.


In [ ]:
results = []
aligned_by_camera = {}
for camera in CAMERAS:
    sam_csv = ANGLE_FILES[camera]
    sync_json = SYNC_FILES[camera]
    if not (sam_csv.exists() and sync_json.exists() and VICON_CSV.exists()):
        print(f"SKIP {camera}: raw SAM angles, reviewed sync, or Vicon reference missing")
        continue
    for side in ("left", "right"):
        prefix = "l" if side == "left" else "r"
        sam_headers = pd.read_csv(sam_csv, nrows=0).columns
        vicon_headers = pd.read_csv(VICON_CSV, nrows=0).columns
        if not any(f"{prefix}_{j}_flex_{m}" in sam_headers
                   for j in ("hip", "knee", "ankle") for m in ("geo", "euler", "smooth")):
            print(f"SKIP {camera}/{side}: no real SAM angle columns")
            continue
        if not any(f"{prefix}_vicon_{j}_flex" in vicon_headers for j in ("hip", "knee", "ankle")):
            print(f"SKIP {camera}/{side}: no independent side-specific Vicon columns")
            continue
        out = RESULTS / SUBJECT / TRIAL / camera / "validation" / side
        metrics = compare_angles(
            angle_csv=sam_csv,
            output_dir=out,
            sync_json=sync_json,
            vicon_csv=VICON_CSV,
            subject=SUBJECT, trial=TRIAL, camera=camera, side=side,
            static_window_ms=STATIC_WINDOW_MS,
            diagnostic_shift_ms=None,
            plot=False,
        )
        results.append(metrics)
        side_frame = pd.read_csv(out / "aligned_angles.csv")
        if camera not in aligned_by_camera:
            aligned_by_camera[camera] = side_frame
        else:
            # The two side-specific comparison runs share the same SAM rows and times.
            # Add independent Vicon columns for the other side only.
            cols_to_add = [c for c in side_frame.columns
                           if c.startswith(prefix + "_vicon_")]
            for col in cols_to_add:
                aligned_by_camera[camera][col] = side_frame[col].to_numpy()

if results:
    summary = pd.concat(results, ignore_index=True)
    display(summary.round(2))
    target = RESULTS / SUBJECT / TRIAL / "validation_metrics_bilateral.csv"
    target.parent.mkdir(parents=True, exist_ok=True)
    summary.to_csv(target, index=False)
    print("Saved", target)
else:
    print("No validation metrics produced: resolve missing input files/columns shown above")


## Bilateral plots
Solid line = SAM, dashed line = Vicon; green = right, red = left. Each camera uses its **own** independently reviewed synchronization offset.


In [ ]:
for camera, frame in aligned_by_camera.items():
    display(plot_joint_overlays(frame,
             title=f"{SUBJECT} {TRIAL} {camera}: bilateral angles", method=METHOD))
    display(plot_joint_errors(frame,
             title=f"{SUBJECT} {TRIAL} {camera}: bilateral errors", method=METHOD))
if len(aligned_by_camera) >= 2:
    display(plot_camera_comparison(aligned_by_camera, joint="knee", method=METHOD))


## Interpretation and limitations
- **Do not re-sync legacy** `*_joint_angles.csv` files that contain `time_ms_aligned` / interpolated Vicon. They are available for exploratory historical comparisons only.
- Current `comparison_engine.py` supports each side independently. This notebook runs it twice then merges the visualizations; it does **not** assume left Vicon equals right Vicon.
- A missing ankle reference is missing data, not zero ankle motion.
- The neutral-offset feature requires a verified standing window; do not use it to conceal systematic definition differences.
- The original notebook's cross-correlation-based diagnostic shifts and `333ms` labelled plots are intentionally excluded from the primary validation.
- Review each candidate sync against a second physical movement to check clock drift.
